In [1]:
import numpy as np
import math
import matplotlib
from sys import path as sys_path
from dn2dem_pos import dn2dem_pos
import warnings

import matplotlib.pyplot as plt
import scipy.io as io

In [2]:
# Simpler example with synthetic data for map testing
# * 23-Nov-2025 IGH

# Setup
warnings.simplefilter('ignore')
matplotlib.rcParams['font.size'] = 16

# Load in the SSWIDL generated response functions
# Was produced by make_aiaresp_forpy.pro (can't escape sswidl that easily....)
trin = io.readsav('aia_tresp_en.dat')

# Get rid of the b in the string name (byte vs utf stuff....)
for i in np.arange(len(trin['channels'])):
    trin['channels'][i] = trin['channels'][i].decode("utf-8")
print(trin['channels'])

# Get the temperature response functions in the correct form for demreg
tresp_logt = np.array(trin['logt'])
nt = len(tresp_logt)
nf = len(trin['tr'][:])
trmatrix = np.zeros((nt, nf))
for i in range(0, nf):
    trmatrix[:, i] = trin['tr'][i]
matplotlib.rcParams['font.size'] = 16

# For some DEM model (i.e. a Gaussian) produce the synthetic DN/s/px for each AIA channel
d1 = 4e22
m1 = 6.5
s1 = 0.15
root2pi = (2. * math.pi) ** 0.5
dem_mod = (d1 / (root2pi * s1)) * np.exp(-(tresp_logt - m1) ** 2 / (2 * s1 ** 2))

# Now work out the DN/s/px
# For AIA responses all are dlogt=0.05
tresp_dlogt = np.full(nt, 0.05)
tc_full = np.zeros([nt, nf])
for i in range(0, nf):
    tc_full[:, i] = dem_mod * trmatrix[:, i] * 10 ** tresp_logt * np.log(10 ** tresp_dlogt)

dn_in = np.sum(tc_full, 0)
print('dn_in: ', dn_in)


['A94' 'A131' 'A171' 'A193' 'A211' 'A335']
dn_in:  [  326.11015384   313.31711081  2663.51032443 11361.19771881
  8700.35613926  1208.36625448]


In [3]:
nx=250
ny=40
# Only input data and error needs the extra dimensions here - TR resp and T binnings as before
dn_in2d=np.zeros([nx,ny,nf])
edn_in2d=np.zeros([nx,ny,nf])

for y in np.arange(ny):
    for x in np.arange(nx):
        # Giving some x,y dependance to our synthetic data so we can see the change in the DEM maps
        # Obviously with real data you wouldn't need to do this
        dn_in2d[x,y,:]=dn_in *(1+(y*1.5/ny)+(x*2./nx))
        # Just use 10% for error as changed data - again with real data use the full error calculation
        edn_in2d[x,y,:]=0.1*dn_in 
        
#  Do slightly differnt binning for these DEMs
t_space=0.05
t_min=5.6
t_max=7.4
logtemps=np.linspace(t_min,t_max,num=int((t_max-t_min)/t_space)+1)
temps=10**logtemps   
        
# Now do the DEM maps calculation           
dem2d,edem2d,elogt2d,chisq2d,dn_reg2d=dn2dem_pos(dn_in2d,edn_in2d,trmatrix,tresp_logt,temps)
# Testing on 8 CPU/thread machine so default should be max_workers=0.5*8=4 

Num CPUs/Threads used: 4


100%|██████████| 100/100 [00:03<00:00, 29.9 x10^2 DEM/s] 


In [4]:
dem2d,edem2d,elogt2d,chisq2d,dn_reg2d=dn2dem_pos(dn_in2d,edn_in2d,trmatrix,tresp_logt,temps,max_workers=8)

Num CPUs/Threads used: 8


100%|██████████| 100/100 [00:02<00:00, 45.3 x10^2 DEM/s] 


In [5]:
dem2d,edem2d,elogt2d,chisq2d,dn_reg2d=dn2dem_pos(dn_in2d,edn_in2d,trmatrix,tresp_logt,temps,max_workers=6)

Num CPUs/Threads used: 6


100%|██████████| 100/100 [00:02<00:00, 36.9 x10^2 DEM/s] 


In [6]:
dem2d,edem2d,elogt2d,chisq2d,dn_reg2d=dn2dem_pos(dn_in2d,edn_in2d,trmatrix,tresp_logt,temps,max_workers=1)

Num CPUs/Threads used: 1


100%|██████████| 100/100 [00:11<00:00, 8.48 x10^2 DEM/s] 
